# ANPR full-pipeline accuracy test (Colab GPU vs local CPU)

This runs the **actual production code** (`PlateDetector` + `PlateTracker` + `PlateReader` + `OcrGate` + `normalize_plate` — the same classes `app/services/pipeline.py` uses), not a rewritten copy, so any difference in results is a real GPU-vs-CPU difference and not drift between two hand-maintained implementations.

**Before running, on your own computer:**
1. From the `backend/` directory, zip the app code (no venv/pycache):
   ```bash
   cd backend
   zip -r app.zip app -x "*__pycache__*"
   ```
2. Also produce the local baseline to compare against:
   ```bash
   python run_pipeline_offline.py --source /path/to/vid1.mp4 --out local_results.csv
   ```
3. Upload to a Google Drive folder (e.g. `anpr_pipeline_test`): `app.zip`, `vechile_plate_yolov8s.pt` (from `backend/models/` — check `PLATE_MODEL_PATH` in `app/config.py` if you're using a different weight), and the same video (`vid1.mp4`).

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
!pip install -q ultralytics opencv-python-headless paddlepaddle-gpu paddleocr==2.7.3 python-dotenv
# numpy installed LAST, forced, no-deps -- so nothing installed above can silently
# overwrite/partially-patch its files afterward (that partial-overwrite is what caused
# the earlier 'No module named numpy.rec' / 'numpy.char' corruption).
!pip install -q --force-reinstall --no-deps "numpy==1.26.4"


**After this cell finishes:** if Colab shows a "restart runtime" prompt, click it / **Runtime -> Restart session**, then re-run this same cell once more in the fresh session. Do this restart at most once, right here — after that, go straight to the mount/stage cell below without running any more `pip install` commands (installing anything else afterward risks silently touching numpy's files again and reintroducing the corruption).

## Mount Drive and unpack the app code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/anpr_pipeline_test'

import os, shutil, zipfile
assert os.path.exists(f'{DRIVE_FOLDER}/app.zip'), "app.zip not found in Drive folder"

shutil.rmtree('/content/backend', ignore_errors=True)
os.makedirs('/content/backend', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE_FOLDER}/app.zip') as z:
    z.extractall('/content/backend')

os.makedirs('/content/backend/models', exist_ok=True)

MODEL_NAME = 'vechile_plate_yolov8s.pt'  # must match PLATE_MODEL_PATH in app/config.py
VIDEO_NAME = 'sample_video.mp4'

shutil.copy(f'{DRIVE_FOLDER}/{MODEL_NAME}', f'/content/backend/models/{MODEL_NAME}')
shutil.copy(f'{DRIVE_FOLDER}/{VIDEO_NAME}', f'/content/{VIDEO_NAME}')

print("App code + model + video staged.")

In [ ]:
import sys
sys.path.insert(0, '/content/backend')

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

# app/config.py reads DEVICE from torch.cuda.is_available() at import time,
# so it will resolve to "cuda" here automatically once the runtime is GPU.

## Run the exact same pipeline logic as `run_pipeline_offline.py`

Same classes, same OCR gating, same validator — imported straight from the unzipped `app` package rather than retyped, so this can't silently diverge from what runs locally.

In [ ]:
import time
import cv2

from app.detection.plate_detector import PlateDetector
from app.ocr.plate_reader import PlateReader
from app.ocr.plate_validator import normalize_plate
from app.services.ocr_gate import OcrGate
from app.tracking.plate_tracker import PlateTracker


def _crop(frame, bbox):
    x1, y1, x2, y2 = bbox
    h, w = frame.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    return frame[y1:y2, x1:x2]


class TrackRecord:
    def __init__(self, vehicle_type, first_frame):
        self.vehicle_type = vehicle_type
        self.first_frame = first_frame
        self.last_frame = first_frame
        self.ocr_attempts = 0
        self.best_text = None
        self.best_confidence = 0.0
        self.readings = set()


detector = PlateDetector()
tracker = PlateTracker(detector)
ocr_reader = PlateReader()
ocr_gate = OcrGate()

# EXPERIMENTAL: testing OCR on GPU after upgrading to paddlepaddle-gpu==3.3.0
# (cu126 build). This downgraded torch's own nvidia-cudnn-cu12 from 9.19.0.56 to
# 9.5.1.17 as a side effect -- if this cell errors (either here or at YOLO
# detection above), that version clash is why; revert to use_gpu=False (the
# known-working CPU fallback) rather than debugging further.
from paddleocr import PaddleOCR
ocr_reader._ocr = PaddleOCR(use_angle_cls=False, lang="en", show_log=False, use_gpu=True)
print("OCR engine set to GPU (experimental).")

VIDEO_PATH = f'/content/{VIDEO_NAME}'
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open {VIDEO_PATH}"

records = {}
frame_rows = []  # one row per detected plate per frame -- bbox coords + timings
frame_id = 0
t_start = time.perf_counter()

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_id += 1

    t_detect = time.perf_counter()
    tracked = tracker.track(frame)
    detect_ms = (time.perf_counter() - t_detect) * 1000

    for plate in tracked:
        record = records.get(plate.track_id)
        if record is None:
            record = TrackRecord(plate.vehicle_type, frame_id)
            records[plate.track_id] = record
        record.last_frame = frame_id

        vx1, vy1, vx2, vy2 = plate.vehicle_bbox if plate.vehicle_bbox else ("", "", "", "")
        px1, py1, px2, py2 = plate.bbox

        crop = _crop(frame, plate.bbox)
        ocr_ran = crop.size > 0 and ocr_gate.should_run(plate.track_id, frame_id, crop)
        ocr_raw_text = ocr_confidence = ocr_validated = ocr_status = ""
        ocr_ms = ""

        if ocr_ran:
            t_ocr = time.perf_counter()
            result = ocr_reader.read(crop)
            ocr_ms = f"{(time.perf_counter() - t_ocr) * 1000:.1f}"

            if result is None:
                ocr_status = "no_text"
            else:
                ocr_gate.record_attempt(plate.track_id, frame_id, result.confidence, crop)
                record.ocr_attempts += 1
                ocr_raw_text = result.text
                ocr_confidence = f"{result.confidence:.3f}"

                normalized = normalize_plate(result.text)
                if normalized is not None:
                    ocr_status = "accepted"
                    ocr_validated = normalized
                    record.readings.add(normalized)
                    if result.confidence > record.best_confidence:
                        record.best_text = normalized
                        record.best_confidence = result.confidence
                else:
                    ocr_status = "rejected"

        frame_rows.append([
            frame_id, plate.track_id, plate.vehicle_type,
            vx1, vy1, vx2, vy2,
            px1, py1, px2, py2,
            f"{plate.confidence:.3f}", f"{detect_ms:.1f}",
            ocr_ran, ocr_raw_text, ocr_confidence, ocr_validated,
            ocr_status, ocr_ms,
        ])

    if frame_id % 100 == 0:
        print(f"  ...{frame_id} frames done")

cap.release()
elapsed = time.perf_counter() - t_start
print(f"\nProcessed {frame_id} frames in {elapsed:.1f}s ({frame_id/elapsed:.1f} fps)")
print(f"Tracks seen: {len(records)}")

## Write results CSVs (same schema as `run_pipeline_offline.py`) and download

Writes both the per-track summary (`colab_results.csv`) and the per-frame detail (`colab_frames.csv`, matching `--frames-out`).

In [ ]:
import csv

FRAMES_OUT_PATH = '/content/colab_frames.csv'
with open(FRAMES_OUT_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        "frame_id", "track_id", "vehicle_type",
        "vehicle_bbox_x1", "vehicle_bbox_y1", "vehicle_bbox_x2", "vehicle_bbox_y2",
        "plate_bbox_x1", "plate_bbox_y1", "plate_bbox_x2", "plate_bbox_y2",
        "plate_confidence", "detect_inference_ms",
        "ocr_ran", "ocr_raw_text", "ocr_confidence", "ocr_validated_plate",
        "ocr_status", "ocr_inference_ms",
    ])
    writer.writerows(frame_rows)
print(f"Wrote {FRAMES_OUT_PATH}")

OUT_PATH = '/content/colab_results.csv'
with open(OUT_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        "track_id", "vehicle_type", "first_frame", "last_frame",
        "ocr_attempts", "best_plate", "best_confidence", "all_readings",
    ])
    for track_id, r in sorted(records.items()):
        writer.writerow([
            track_id, r.vehicle_type, r.first_frame, r.last_frame,
            r.ocr_attempts, r.best_text or "", f"{r.best_confidence:.3f}",
            ";".join(sorted(r.readings)),
        ])
print(f"Wrote {OUT_PATH}")

from google.colab import files
files.download(OUT_PATH)
files.download(FRAMES_OUT_PATH)

## Compare against your local run

Back on your own computer, with `local_results.csv`/`local_frames.csv` (from `run_pipeline_offline.py`) and the downloaded `colab_results.csv`/`colab_frames.csv` both in `backend/`:
```bash
python compare_pipeline_runs.py local_results.csv colab_results.csv
```
This reports which validated plate readings both runs agree on, which show up in only one, and the confidence delta on the ones they agree on — i.e. whether GPU changes *what* gets read, not just how fast it runs.

The `*_frames.csv` files are for drilling into a specific frame/track — bbox coordinates, per-frame detection confidence, and OCR inference time — if the summary comparison flags a mismatch worth inspecting.